# 04 · Consultas de negócio

Respostas às seis perguntas do PDF usando `display()`, sem bibliotecas de gráficos.

As consultas utilizam a fato de filmes lançados e suas dimensões e pontes. Valores ausentes não são convertidos em zero. As somas financeiras são acompanhadas da quantidade de filmes com valor conhecido, pois a base possui muitos orçamentos e receitas ausentes. BRL usa a cotação de compra escolhida na Silver.

### Parâmetros e leitura da Gold

In [ ]:
# Padronização de nomes: <catalog>.<camada>.<tabela>, tudo em snake_case
catalog = "cine_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"
landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/landing"

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_fact = spark.table(f"{gold_schema}.fact_movies_performance")
df_movies = spark.table(f"{gold_schema}.dim_movies")
df_genres = spark.table(f"{gold_schema}.dim_genres")
df_people = spark.table(f"{gold_schema}.dim_people")
df_companies = spark.table(f"{gold_schema}.dim_companies")
df_bridge_genre = spark.table(f"{gold_schema}.bridge_movie_genre")
df_bridge_person = spark.table(f"{gold_schema}.bridge_movie_person")
df_bridge_company = spark.table(f"{gold_schema}.bridge_movie_company")
df_desempenho = df_fact.join(df_movies, "sk_movie_id", "inner")

## 1. Qual é a receita total em R$?

In [ ]:
df_receita_total = df_desempenho.agg(
    F.sum("receita_brl").alias("receita_total_brl"),
    F.count("*").alias("qtd_filmes_lancados"),
    F.count("receita_brl").alias("qtd_filmes_com_receita"),
    F.sum(F.when(F.col("receita_brl").isNull(), 1).otherwise(0)).alias("qtd_filmes_sem_receita"),
)
display(df_receita_total)

Para a leitura literal de **todos os filmes da base**, incluindo os ainda não lançados que ficam fora da fato, o total do catálogo é calculado abaixo com a Silver financeira deduplicada. Não somamos resultados de populações diferentes.

In [ ]:
df_receita_catalogo = (df_movies.select("id_filme")
    .join(spark.table(f"{silver_schema}.tb_financeiro_filmes").select("id_filme", "receita_brl"), "id_filme", "left")
    .agg(F.sum("receita_brl").alias("receita_total_catalogo_brl"),
         F.count("*").alias("qtd_filmes_catalogo"),
         F.count("receita_brl").alias("qtd_filmes_com_receita")))
display(df_receita_catalogo)

## 2. Quais são os 5 filmes com maior popularidade?

`revisar_origem` sinaliza a combinação de popularidade com aparência de ano e textos incompatíveis nos dois campos IMDb da origem. Esses valores não são corrigidos por suposição. O ranking é calculado sobre os valores disponíveis, mas linhas sinalizadas precisam de confirmação na fonte antes do uso em decisões.

In [ ]:
df_top_popularidade = (df_desempenho.filter(F.col("popularidade").isNotNull())
    .select("titulo", "popularidade", "id_filme")
    .orderBy(F.desc("popularidade"), "id_filme").limit(5))
df_sinalizacao = spark.table(f"{silver_schema}.tb_metricas_engajamento").select("id_filme", "revisar_origem")
display(df_top_popularidade.join(df_sinalizacao, "id_filme", "left")
        .orderBy(F.desc("popularidade"), "id_filme")
        .select("titulo", "popularidade", "revisar_origem"))

## 3. Quantos filmes cada gênero possui?

In [ ]:
# Cada vínculo filme/gênero é único na ponte. Um filme pode aparecer em vários gêneros.
# Inclui todo o catálogo dimensional, pois a pergunta não restringe a lançados.
df_filmes_genero = (df_genres.join(df_bridge_genre, "sk_genre_id", "left")
    .groupBy("sk_genre_id", "nome_genero")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_filmes"))
    .orderBy(F.desc("qtd_filmes"), "nome_genero"))
display(df_filmes_genero.select("nome_genero", "qtd_filmes"))

## 4. Quais são os 10 filmes de maior receita?

In [ ]:
# RANK atribui a mesma posição às receitas empatadas. O ID apenas ordena a
# apresentação: não entra na janela, para não desfazer os empates.
janela_ranking = Window.orderBy(F.desc("receita_usd"))
df_top_receita = (df_desempenho.filter(F.col("receita_usd").isNotNull())
    .withColumn("posicao", F.rank().over(janela_ranking))
    .orderBy("posicao", "id_filme").limit(10))
display(df_top_receita.select("posicao", "titulo", "receita_usd", "receita_brl"))

### Referência temporal para as perguntas 5 e 6
A referência é a maior data de lançamento **realizado** da base: status Lançado, data não nula e não futura. Os intervalos são inclusivos, de 24 e 60 meses antes dessa referência até a própria referência.

In [ ]:
data_referencia = (df_movies.filter((F.col("status_filme") == "Lançado")
                                    & (F.col("data_lancamento") <= F.current_date()))
                    .agg(F.max("data_lancamento").alias("data_referencia")).first()["data_referencia"])
if data_referencia is None:
    raise ValueError("A base não contém um lançamento realizado com data válida.")
display(spark.range(1).select(
    F.lit(data_referencia).alias("data_referencia"),
    F.add_months(F.lit(data_referencia), -24).alias("inicio_ultimos_2_anos"),
    F.add_months(F.lit(data_referencia), -60).alias("inicio_ultimos_5_anos"),
))

## 5. Qual ator mais participou de filmes nos últimos 2 anos?

In [ ]:
df_filmes_2_anos = df_desempenho.filter(F.col("data_lancamento").between(
    F.add_months(F.lit(data_referencia), -24), F.lit(data_referencia)))
df_atores = (df_filmes_2_anos.select("sk_movie_id")
    .join(df_bridge_person, "sk_movie_id", "inner")
    .join(df_people.filter(F.col("tipo_pessoa") == "Ator"), "sk_person_id", "inner")
    .groupBy("sk_person_id", "nome_pessoa")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_participacoes"))
    .withColumn("posicao", F.rank().over(Window.orderBy(F.desc("qtd_participacoes")))))
# Mostrar todos os líderes em caso de empate.
display(df_atores.filter(F.col("posicao") == 1).select("nome_pessoa", "qtd_participacoes").orderBy("nome_pessoa"))

## 6. Qual produtora teve maior lucro nos últimos 5 anos?
Usamos lucro em BRL. Cada produtora recebe o lucro integral dos filmes dos quais participou, pois não existe participação contratual na base para fazer um rateio. Por isso, o total por produtora não deve ser somado entre produtoras. Filmes sem orçamento ou receita não têm lucro conhecido.

In [ ]:
df_filmes_5_anos = df_desempenho.filter(F.col("data_lancamento").between(
    F.add_months(F.lit(data_referencia), -60), F.lit(data_referencia)))
df_lucro_produtoras = (df_filmes_5_anos.select("sk_movie_id", "lucro_usd", "lucro_brl")
    .join(df_bridge_company, "sk_movie_id", "inner")
    .join(df_companies, "sk_company_id", "inner")
    .groupBy("sk_company_id", "nome_produtora")
    .agg(F.sum("lucro_usd").alias("lucro_total_usd"), F.sum("lucro_brl").alias("lucro_total_brl"),
         F.countDistinct("sk_movie_id").alias("qtd_filmes"),
         F.count("lucro_brl").alias("qtd_filmes_com_lucro_conhecido"))
    .filter(F.col("qtd_filmes_com_lucro_conhecido") > 0)
    .withColumn("posicao", F.rank().over(Window.orderBy(F.desc("lucro_total_brl")))))
display(df_lucro_produtoras.filter(F.col("posicao") == 1)
        .select("nome_produtora", "lucro_total_usd", "lucro_total_brl", "qtd_filmes", "qtd_filmes_com_lucro_conhecido")
        .orderBy("nome_produtora"))